## Preprocessing

In [22]:
import numpy as np

docs = ['go india',
		'india india',
		'hip hip hurray',
		'jeetega bhai jeetega india jeetega',
		'bharat mata ki jai',
		'kohli kohli',
		'sachin sachin',
		'dhoni dhoni',
		'modi ji ki jai',
		'inquilab zindabad']

In [23]:
import tensorflow
from tensorflow import keras
from keras.preprocessing.text import Tokenizer

In [24]:
tokenizer = Tokenizer(oov_token='<nothing>')
# oov_token -> out of vocabulary -> while testing if we got a word which is not present in vocabulary thins we can replace that word with this

tokenizer.fit_on_texts(docs)

In [25]:
tokenizer.word_index # All unique words from dataset

{'<nothing>': 1,
 'india': 2,
 'jeetega': 3,
 'hip': 4,
 'ki': 5,
 'jai': 6,
 'kohli': 7,
 'sachin': 8,
 'dhoni': 9,
 'go': 10,
 'hurray': 11,
 'bhai': 12,
 'bharat': 13,
 'mata': 14,
 'modi': 15,
 'ji': 16,
 'inquilab': 17,
 'zindabad': 18}

In [26]:
tokenizer.word_counts

OrderedDict([('go', 1),
             ('india', 4),
             ('hip', 2),
             ('hurray', 1),
             ('jeetega', 3),
             ('bhai', 1),
             ('bharat', 1),
             ('mata', 1),
             ('ki', 2),
             ('jai', 2),
             ('kohli', 2),
             ('sachin', 2),
             ('dhoni', 2),
             ('modi', 1),
             ('ji', 1),
             ('inquilab', 1),
             ('zindabad', 1)])

In [27]:
tokenizer.document_count # Number of input sentences

10

In [28]:
sequences = tokenizer.texts_to_sequences(docs)
sequences

[[10, 2],
 [2, 2],
 [4, 4, 11],
 [3, 12, 3, 2, 3],
 [13, 14, 5, 6],
 [7, 7],
 [8, 8],
 [9, 9],
 [15, 16, 5, 6],
 [17, 18]]

In [29]:
from keras.utils import pad_sequences
sequences = pad_sequences(sequences,padding='post')
sequences

array([[10,  2,  0,  0,  0],
       [ 2,  2,  0,  0,  0],
       [ 4,  4, 11,  0,  0],
       [ 3, 12,  3,  2,  3],
       [13, 14,  5,  6,  0],
       [ 7,  7,  0,  0,  0],
       [ 8,  8,  0,  0,  0],
       [ 9,  9,  0,  0,  0],
       [15, 16,  5,  6,  0],
       [17, 18,  0,  0,  0]], dtype=int32)

---

## Training SimpleRNN model using integer Encoding

In [30]:
from keras.datasets import imdb
from keras import Sequential
from keras.layers import Input, Dense, SimpleRNN

In [81]:
(X_train, y_train), (X_test, y_test) = imdb.load_data()

In [32]:
len(X_train[0])

218

In [33]:
len(X_train[1])

189

In [82]:
X_train = pad_sequences(X_train,padding='post', maxlen=500)
X_test = pad_sequences(X_test,padding='post', maxlen=500)
# maxlen -> Just using because no unique wirds in x_train and x_test are not same

In [54]:
X_test.shape

(25000, 500)

In [55]:
model = Sequential()

model.add(Input(shape = (X_train.shape[1], 1,))) # Shape is in the form of -> (total_timesteps, no_features_passing_at each_time: There is only 1 Feature because we are passing only one word at a time)
model.add(SimpleRNN(units = 32, return_sequences = False)) # return_sequences = True -> You can return an outout at each node -> Certain applications this is useful
model.add(Dense(1,activation='sigmoid'))

model.summary()

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 simple_rnn_5 (SimpleRNN)    (None, 32)                1088      
                                                                 
 dense_6 (Dense)             (None, 1)                 33        
                                                                 
Total params: 1121 (4.38 KB)
Trainable params: 1121 (4.38 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [56]:
model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [57]:
model.fit(X_train, y_train, epochs=5, validation_data = (X_test,y_test))

Epoch 1/5
782/782 [==============================] - 63s 79ms/step - loss: 0.6933 - accuracy: 0.5008 - val_loss: 0.6932 - val_accuracy: 0.5000
Epoch 2/5
782/782 [==============================] - 63s 81ms/step - loss: 0.6941 - accuracy: 0.4943 - val_loss: 0.6935 - val_accuracy: 0.5000
Epoch 3/5
782/782 [==============================] - 61s 78ms/step - loss: 0.6945 - accuracy: 0.4948 - val_loss: 0.6940 - val_accuracy: 0.5000
Epoch 4/5
782/782 [==============================] - 60s 77ms/step - loss: 0.6946 - accuracy: 0.4953 - val_loss: 0.6932 - val_accuracy: 0.5000
Epoch 5/5
782/782 [==============================] - 64s 82ms/step - loss: 0.6939 - accuracy: 0.5014 - val_loss: 0.6972 - val_accuracy: 0.5000


---

## Training SimpleRNN model using Embeddings

In [84]:
from keras.layers import Embedding

model = Sequential()
model.add(Embedding(input_dim = 90000, output_dim = 5, input_length = 500))
'''
Embedding is used after Padding (IMP)

input_dim -> no. of unique words in the dataset
why only 90000 -> The model will take as input an integer matrix of size (batch, input_length), and the largest integer (i.e. word index) in the input should be no larger than 999 (vocabulary size).
output_dim -> no. of output features from sparce matrix(padded one)
input_lenght -> length of features in each row/sentence after padding
'''
model.add(SimpleRNN(units = 32, return_sequences=False))
model.add(Dense(units=1, activation='sigmoid'))

model.summary()

Model: "sequential_11"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_5 (Embedding)     (None, 500, 5)            450000    
                                                                 
 simple_rnn_10 (SimpleRNN)   (None, 32)                1216      
                                                                 
 dense_11 (Dense)            (None, 1)                 33        
                                                                 
Total params: 451249 (1.72 MB)
Trainable params: 451249 (1.72 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [85]:
model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [86]:
model.fit(x = X_train, y = y_train, validation_data = (X_test, y_test), epochs = 5)

Epoch 1/5
782/782 [==============================] - 97s 122ms/step - loss: 0.6940 - accuracy: 0.5031 - val_loss: 0.6936 - val_accuracy: 0.5016
Epoch 2/5
782/782 [==============================] - 92s 118ms/step - loss: 0.6935 - accuracy: 0.5079 - val_loss: 0.6930 - val_accuracy: 0.5020
Epoch 3/5
782/782 [==============================] - 91s 117ms/step - loss: 0.6922 - accuracy: 0.5168 - val_loss: 0.6945 - val_accuracy: 0.5002
Epoch 4/5
782/782 [==============================] - 93s 119ms/step - loss: 0.6912 - accuracy: 0.5182 - val_loss: 0.6930 - val_accuracy: 0.5016
Epoch 5/5
782/782 [==============================] - 95s 122ms/step - loss: 0.6888 - accuracy: 0.5267 - val_loss: 0.6943 - val_accuracy: 0.5022
